# ScienceQA Hybrid RAG Pipeline Evaluation & Visualization

This notebook evaluates the ScienceQA Hybrid Retrieval-Augmented Generation (RAG) pipeline (BERT + TF-IDF) on the test set, reports accuracy by grade and subject, and visualizes the results.

In [1]:
# 1. Import Required Libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json
import os

## 2. Load Model and Data
Load the custom BERT model, TF-IDF vectorizer, and ScienceQA datasets.

In [ ]:
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "models")))
from custom_bert import CustomBERTEmbedding

VOCAB_SIZE = 30522
EMBED_DIM = 256
NUM_HEADS = 8
NUM_LAYERS = 6
MAX_SEQ_LENGTH = 128

# Load model
model = CustomBERTEmbedding(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, NUM_LAYERS, MAX_SEQ_LENGTH)
model.load_state_dict(torch.load(os.path.join("..", "models", "custom_bert.pth"), map_location=torch.device('cpu')))
model.eval()

# Load data
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
train_path = os.path.join(data_dir, "train.json")
test_path = os.path.join(data_dir, "test.json")
with open(train_path, 'r', encoding='utf-8') as f:
    data_train = json.load(f)
with open(test_path, 'r', encoding='utf-8') as f:
    data_test = json.load(f)

print(f"Train samples: {len(data_train)} | Test samples: {len(data_test)}")

/Users/jessicalim/miniconda3/envs/ml_project/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/var/folders/7n/sdnmyc4j33q0y4lz8rx2m3wc0000gn/T/ipykernel_94847/1172668983.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowl

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jessicalim/Desktop/UNSW/COMP9444/scienceqa-9444/data/ScienceQA_train.json'

## 3. Load or Compute BERT and TF-IDF Embeddings
Load precomputed BERT and TF-IDF embeddings for lectures, or compute and save them if not present.

In [ ]:
def simple_tokenizer(text, vocab_size=VOCAB_SIZE, max_seq_length=MAX_SEQ_LENGTH):
    tokens = text.lower().split()
    ids = [abs(hash(token)) % vocab_size for token in tokens]
    if len(ids) < max_seq_length:
        ids += [0] * (max_seq_length - len(ids))
    else:
        ids = ids[:max_seq_length]
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0)

def embed_text(model, text):
    input_ids = simple_tokenizer(text)
    with torch.no_grad():
        emb = model(input_ids)
        emb_vec = emb.mean(dim=1).squeeze().cpu().numpy()
    return emb_vec

def batch_embed_lectures(model, data, save_path):
    embs = []
    for item in tqdm(data, desc="Embedding lectures (BERT)"):
        lecture = item.get('lecture', '')
        embs.append(embed_text(model, lecture))
    embs = np.stack(embs)
    np.save(save_path, embs)
    print(f"Saved {len(embs)} BERT embeddings to {save_path}")

lectures = [item.get('lecture', '') for item in data_train]
bert_emb_path = os.path.join(data_dir, "lecture_embeddings.npy")
tfidf_path = os.path.join(data_dir, "lecture_tfidf.npz")
tfidf_vocab_path = os.path.join(data_dir, "tfidf_vocab.json")

# BERT embeddings
if not os.path.exists(bert_emb_path):
    batch_embed_lectures(model, data_train, bert_emb_path)
bert_embs = np.load(bert_emb_path)

# TF-IDF embeddings
if not os.path.exists(tfidf_path):
    vectorizer = TfidfVectorizer(max_features=4096)
    tfidf_matrix = vectorizer.fit_transform(lectures)
    from scipy import sparse
    sparse.save_npz(tfidf_path, tfidf_matrix)
    with open(tfidf_vocab_path, "w") as f:
        json.dump(vectorizer.vocabulary_, f)
else:
    from scipy import sparse
    tfidf_matrix = sparse.load_npz(tfidf_path)
    with open(tfidf_vocab_path, "r") as f:
        vocab = json.load(f)
    vectorizer = TfidfVectorizer(vocabulary=vocab)

print(f"Loaded BERT embeddings: {bert_embs.shape}")
print(f"Loaded TF-IDF matrix: {tfidf_matrix.shape}")

## 4. Hybrid Retrieval and Evaluation on Test Set
Run hybrid retrieval (BERT + TF-IDF) for each test question and collect results.

In [ ]:
results = []
for item in tqdm(data_test, desc="Hybrid retrieval on test set"):
    query = item.get('question', '')
    gt_answer = item.get('answer', '')
    grade = item.get('grade', 'Unknown')
    subject = item.get('subject', 'Unknown')
    # BERT retrieval
    query_emb = embed_text(model, query)
    bert_sims = np.dot(bert_embs, query_emb) / (np.linalg.norm(bert_embs, axis=1) * np.linalg.norm(query_emb) + 1e-8)
    # TF-IDF retrieval
    tfidf_query = vectorizer.transform([query])
    tfidf_sims = cosine_similarity(tfidf_matrix, tfidf_query).flatten()
    # Hybrid: weighted sum (0.5 BERT + 0.5 TF-IDF)
    hybrid_sims = 0.5 * bert_sims + 0.5 * tfidf_sims
    top_idx = np.argmax(hybrid_sims)
    hit = data_train[top_idx]
    # For this notebook, just check if the top retrieved lecture contains the ground truth answer (string match)
    lecture = hit.get('lecture', '')
    is_correct = str(gt_answer).strip().lower() in lecture.strip().lower()
    results.append({
        'question': query,
        'ground_truth': gt_answer,
        'retrieved_lecture': lecture,
        'grade': grade,
        'subject': subject,
        'is_correct': is_correct
    })
results_df = pd.DataFrame(results)
results_df.head()

## 5. Calculate Accuracy by Grade and Subject
Aggregate the results to compute accuracy for each grade and each subject.

In [ ]:
# Accuracy by grade
acc_by_grade = results_df.groupby('grade')['is_correct'].mean().sort_index()
# Accuracy by subject
acc_by_subject = results_df.groupby('subject')['is_correct'].mean().sort_values(ascending=False)

print("Accuracy by grade:")
print(acc_by_grade)
print("\nAccuracy by subject:")
print(acc_by_subject)

## 6. Visualize Accuracy by Grade
Create a bar plot showing accuracy for each grade using matplotlib or seaborn.

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=acc_by_grade.index, y=acc_by_grade.values, palette="Blues_d")
plt.title("Hybrid RAG Accuracy by Grade")
plt.xlabel("Grade")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

## 7. Visualize Accuracy by Subject
Create a bar plot showing accuracy for each subject using matplotlib or seaborn.

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x=acc_by_subject.index, y=acc_by_subject.values, palette="Greens_d")
plt.title("Hybrid RAG Accuracy by Subject")
plt.xlabel("Subject")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.show()